<a href="https://colab.research.google.com/github/annaannaR/NOTEBOOKS-FROM-SCHOOL/blob/main/AMS/Machine%20Learning/PART_3/MLP3_2_Clustering_github.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

import matplotlib.pyplot as plt
import seaborn as sns

sns.set()

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

#K-means clustering

K-Means clustering is a type of unsupervised learning which means that we do not have any pre-defined labels in our data, and we are not trying to make predictions.

Our goal is to find groups in our data such that the data points are relatively similar to each other within each group.

Like discussed in the class, the algorithm starts with some initial estimate of where each of the centroids could be for each of the clusters. These initial values could for example be chosen randomly.

Next, the algorithm will cycle through these next two steps until the centroid positions don’t change anymore:

* Each data point is assigned to a cluster closest to them;
* Once all data points belong to a single cluster, the mean of all data points within that cluster is calculated and this becomes the new centroid.

What we did not discuss so far is how to define the optimal value of K.
One method to define this value is using the Elbow method, which we will go through below.

## Customer data

We will be using data about mall customers. Download it from the datasets folder, and upload it to your own datasets-folder in Drive (if you haven't already). [Download here](https://drive.google.com/drive/folders/1jgzUGTqbtGuR1NkrtG1vT5edDgES32Id?usp=sharing)

This is customer data containing 4 variables:

* CustomerID
* Residence (urban/suburban)
* Age
* Annual income (x 1000 $)


We are going to explore the data using some visualisations and then train a K-Means clustering model, optimising the number of clusters using the elbow method.

## Data exploration

In [ ]:
# Import data
df = pd.read_csv("/content/drive/MyDrive/datasets/Mall_Customers.csv", delimiter=';')
df.head()

The CustomerID doesn't really add value (almost the same as the index), so we will drop it. Also, we are only going to use numeric columns, so drop the Residence column as well.

In [ ]:
df.drop(columns = ["CustomerID", "Residence"], inplace=True)

In [ ]:
# Look at some statistics of the (numeric) columns
df.describe()

We see we only have 200 rows, and not a lot of variables. But still, let's also take a look at the scatterplots:

In [ ]:
# Check correlations and distribution using scatter-plot
plt.figure(figsize = (15, 6))
sns.pairplot(df, height = 3)
plt.show()

## Data transformation
We see all the data here is in a relatively similar scale (0-100), but a little skewed, so we will still standardize the dataset (also just good practice).

In [ ]:
# Scale the numerical features
scaler = StandardScaler()
scaled_features = scaler.fit_transform(df)

# Turn back into dataframe
scaled_df = pd.DataFrame({"scaled_age": scaled_features[:, 0], "scaled_income": scaled_features[:, 1]})
scaled_df.head()

Now our data is ready for K-Means. All we need to do next is determine the optimal number of clusters. We use the elbow method.

## Elbow method

First let's introduce the term Cluster inertia, also called distortion.

**Cluster inertia** = the within-cluster Sum of Squared Error (SSE) for each cluster.

Therefore the smaller the inertia, the denser the cluster (closer together all the points are).

The idea behind the elbow method is to identify the value of k where the inertia (distortion) begins to decrease most slowly as we keep increasing k. Thus increasing k from this point does not reduce the Sum of Squared error that much anymore.

This is the point where the elbow occurs (hence the name, elbow plot).

Let's try it out, testing with 10 clusters:

In [ ]:
# Calculate inertia
inertia = []
for n in range(1 , 11):
    km = (KMeans(n_clusters = n, init='random', random_state= 1))
    km.fit(scaled_df)
    inertia.append(km.inertia_)

In [ ]:
# Plot the values in a matplotlib-figure
plt.figure(figsize = (15 ,6))
plt.plot(np.arange(1 , 11) , inertia, 'o')
plt.plot(np.arange(1 , 11) , inertia, '-')
plt.legend()
plt.xlabel('Number of Clusters') , plt.ylabel('Inertia')
plt.show()

What we are looking for is the ‘elbow’ in the chart. At what point does the line look like it is first bending?

In this case the optimal number of clusters seems to be k=3.

## Modeling K-means

Now that we have selected the optimal number of clusters, we proceed with the modeling.

We don't have to train_test_split(), our divide into features/target, since we are dealing with an Unsupervised learning problem.

In [ ]:
# Just like other models, first initialize, giving the parameters to use
km = (KMeans(n_clusters = 3, init='random', random_state= 1))

# Then fit the model to the (numerical) data
km.fit(scaled_df)

We can look at the results by seeing the clusters the datapoints got assigned to. And we can even get the 'coordinates' of the final cluster centroids.

In [ ]:
# Get and check clusters
clusters = km.labels_
print("Datapoints belong to cluster-number:", clusters)

# Get and check centroids
centroids = km.cluster_centers_
print("\nCentroids are:", centroids)

It will make more sense using a visualisation:

In [ ]:
plt.figure(figsize = (9,6))
# Plot clusters
sns.scatterplot(x = scaled_df.iloc[:, 0], y = scaled_df.iloc[:, -1], hue = clusters, palette = "Dark2")
# Plot cluster-centroids
sns.scatterplot(x = centroids[:, 0], y = centroids[:, 1], color = "black", marker = "x", s = 200)
# Titles
plt.ylabel("Income")
plt.xlabel("Age")
plt.title("Data Points Allocated to Clusters with corresponding Centroids")
plt.show()

We now have created 3 customer segments, depending on the Age and Income of the customers. These could be used in some way in the business strategy of the mall.

## Assignment 3.2

Now try to create clusters yourself, using the K-means algorithm. Use the following randomly created dataset:

In [ ]:
from sklearn.datasets import make_blobs
dataset, classes = make_blobs(n_samples=500, n_features=3, cluster_std=2.5, random_state=1)
data = pd.DataFrame(dataset, columns=['var1', 'var2', 'var3'])
data.head()

**Question 1**

Explore the data, use the Elbow-method to define the optimal number of clusters, and model and visualize these clusters.